In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 10:37:25 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/03/03 10:37:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 10:37:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 10:37:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
simpleData = (("Java",4000,5), \
    ("Numpy", 4600,10),  \
    ("Polars", 4100,15),   \
    ("Powershell", 4500,15),   \
    ("PySpark", 3000,20),  \
  )
columns= ["CourseName", "fee", "discount"]

df = spark.createDataFrame(data = simpleData, schema = columns)
df.show(truncate=False)

+----------+----+--------+
|CourseName|fee |discount|
+----------+----+--------+
|Java      |4000|5       |
|Numpy     |4600|10      |
|Polars    |4100|15      |
|Powershell|4500|15      |
|PySpark   |3000|20      |
+----------+----+--------+



Функция pyspark.sql.DataFrame.transform() используется для создания цепочки пользовательских преобразований, и эта функция возвращает новый DataFrame после применения указанных преобразований.

Эта функция всегда возвращает то же количество строк, которое существует во входном PySpark DataFrame.

---
**Синтаксис**

Ниже приведен синтаксис функции pyspark.sql.DataFrame.transform()

DataFrame.transform(func: Callable[[…], DataFrame], *args: Any, **kwargs: Any) ->

pyspark.sql.dataframe.DataFrame

---
В приведенном ниже фрагменте я создал три пользовательских преобразования для применения к DataFrame. Эти преобразования - не что иное, как функции Python, которые берут DataFrame, применяют к нему некоторые изменения и возвращают новый DataFrame.

to_upper_str_columns() - Эта функция преобразует столбец CourseName в верхний регистр и обновляет этот же столбец.

reduce_price() - Эта функция принимает аргумент, уменьшает значение из платы и создает новый столбец.

apply_discount() - Создает новый столбец со скидкой.

In [7]:
# Custom transformation 1
def to_upper_str_columns(df):
    return df.withColumn("CourseName",F.upper(df.CourseName))

# Custom transformation 2
def reduce_price(df,reduceBy):
    return df.withColumn("new_fee",df.fee - reduceBy)

# Custom transformation 3
def apply_discount(df):
    return df.withColumn("discounted_fee",  \
             df.new_fee - (df.new_fee * df.discount) / 100)

Теперь давайте соединим эти пользовательские функции в цепочку и запустим их с помощью функции PySpark DataFrame transform().

In [8]:
# PySpark transform() Usage
df2 = df.transform(to_upper_str_columns) \
        .transform(reduce_price,1000) \
        .transform(apply_discount)
df2.show()

+----------+----+--------+-------+--------------+
|CourseName| fee|discount|new_fee|discounted_fee|
+----------+----+--------+-------+--------------+
|      JAVA|4000|       5|   3000|        2850.0|
|     NUMPY|4600|      10|   3600|        3240.0|
|    POLARS|4100|      15|   3100|        2635.0|
|POWERSHELL|4500|      15|   3500|        2975.0|
|   PYSPARK|3000|      20|   2000|        1600.0|
+----------+----+--------+-------+--------------+

